In [1]:
from pyspark.sql.session import SparkSession

spark = (
    SparkSession
    .builder
    .appName("spark-type")
    .master("local[*]")
    .config("spark.port.maxRetries", "0")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

sc = spark.sparkContext


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/suanki/Desktop/pyspark-env/lib/python3.13/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/11/03 19:37:05 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
df = ( spark
      .read
      .format("json")
      .load("../data/flight-data/json/2015-summary.json")
)


df.show()

+--------------------+-------------------+-----+
|   DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count|
+--------------------+-------------------+-----+
|       United States|            Romania|   15|
|       United States|            Croatia|    1|
|       United States|            Ireland|  344|
|               Egypt|      United States|   15|
|       United States|              India|   62|
|       United States|          Singapore|    1|
|       United States|            Grenada|   62|
|          Costa Rica|      United States|  588|
|             Senegal|      United States|   40|
|             Moldova|      United States|    1|
|       United States|       Sint Maarten|  325|
|       United States|   Marshall Islands|   39|
|              Guyana|      United States|   64|
|               Malta|      United States|    1|
|            Anguilla|      United States|   41|
|             Bolivia|      United States|   30|
|       United States|           Paraguay|    6|
|             Algeri

In [3]:
df.printSchema()


root
 |-- DEST_COUNTRY_NAME: string (nullable = true)
 |-- ORIGIN_COUNTRY_NAME: string (nullable = true)
 |-- count: long (nullable = true)



In [4]:
schema = df.schema
schema

StructType([StructField('DEST_COUNTRY_NAME', StringType(), True), StructField('ORIGIN_COUNTRY_NAME', StringType(), True), StructField('count', LongType(), True)])

A schema is a StructType made up of a number of fields, StructFields, that have a name,
type, a Boolean flag which specifies whether that column can contain missing or null values

In [5]:
from pyspark.sql.types import StructType, StructField, StringType, LongType


manual_schema = StructType([
    
        StructField("DEST_COUNTRY_NAME", StringType(), True),
        StructField("ORIGIN_COUNTRY_NAME", StringType(), True),
        StructField("count", LongType(), True)
        
]
)


df = (spark
      .read
      .format('json')
      .schema(manual_schema)
      .load("../data/flight-data/json/2015-summary.json")
)


df.show()

+--------------------+-------------------+-----+
|   DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count|
+--------------------+-------------------+-----+
|       United States|            Romania|   15|
|       United States|            Croatia|    1|
|       United States|            Ireland|  344|
|               Egypt|      United States|   15|
|       United States|              India|   62|
|       United States|          Singapore|    1|
|       United States|            Grenada|   62|
|          Costa Rica|      United States|  588|
|             Senegal|      United States|   40|
|             Moldova|      United States|    1|
|       United States|       Sint Maarten|  325|
|       United States|   Marshall Islands|   39|
|              Guyana|      United States|   64|
|               Malta|      United States|    1|
|            Anguilla|      United States|   41|
|             Bolivia|      United States|   30|
|       United States|           Paraguay|    6|
|             Algeri

### Columns and Expressions
- Columns in Spark are similar to columns in a spreadsheet, R dataframe, or pandas DataFrame.You can select, manipulate, and remove columns from DataFrames and these operations are represented as expressions.

- An expression is
a set of transformations on one or more values in a record in a DataFrame.

In [6]:
## accesing columns
t = df.columns

t, type(t)

(['DEST_COUNTRY_NAME', 'ORIGIN_COUNTRY_NAME', 'count'], list)

### **Records and Rows**

each row in a DataFrame is a single record. Spark represents this record as an object of
type Row. Spark manipulates Row objects using column expressions in order to produce usable
values. Row objects internally represent arrays of bytes

In [7]:
result = df.first()

type(result), result

(pyspark.sql.types.Row,
 Row(DEST_COUNTRY_NAME='United States', ORIGIN_COUNTRY_NAME='Romania', count=15))

**Rows themselves do not have**
schemas. This means that if you create a Row manually, you must specify the values in the same
order as the schema of the DataFrame to which they might be appended

In [8]:
from pyspark.sql import Row

single_record = Row("Hello", None, 1, False)
multiple_row = [Row("Hello", None, 1, False), Row("Hello1", None, 1, False), Row("Hello", None, 1, False)]


In [9]:
single_record

<Row('Hello', None, 1, False)>

In [10]:
multiple_row

[<Row('Hello', None, 1, False)>,
 <Row('Hello1', None, 1, False)>,
 <Row('Hello', None, 1, False)>]

In [11]:
single_record(0)

Row(Hello=0)

In [12]:
single_record[2]

1

In [13]:
multiple_row[0]

<Row('Hello', None, 1, False)>

In [14]:
multiple_row[1]

<Row('Hello1', None, 1, False)>

### DataFrame Transformations

- We can add rows or columns
- We can remove rows or columns
- We can transform a row into a column (or vice versa)
- We can change the order of rows based on the values in columns


In [15]:
### creating a dataframe

df = (
    spark
    .read
    .format("csv")
    .option("header","true")
    .load("../data/flight-data/csv/")
)

df.show(), df.count()

+--------------------+-------------------+-----+
|   DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count|
+--------------------+-------------------+-----+
|       United States|            Romania|    1|
|       United States|            Ireland|  264|
|       United States|              India|   69|
|               Egypt|      United States|   24|
|   Equatorial Guinea|      United States|    1|
|       United States|          Singapore|   25|
|       United States|            Grenada|   54|
|          Costa Rica|      United States|  477|
|             Senegal|      United States|   29|
|       United States|   Marshall Islands|   44|
|              Guyana|      United States|   17|
|       United States|       Sint Maarten|   53|
|               Malta|      United States|    1|
|             Bolivia|      United States|   46|
|            Anguilla|      United States|   21|
|Turks and Caicos ...|      United States|  136|
|       United States|        Afghanistan|    2|
|Saint Vincent and..

(None, 1502)

In [16]:
df.createOrReplaceTempView("dfTable")

In [17]:
## We can also create DataFrames on the fly 
# by taking a set of rows and converting them to a DataFrame.

from pyspark.sql.types import StructField, StructType, StringType, IntegerType

manual_schema = StructType(
    [
        StructField("name",StringType(),True),
        StructField("age",IntegerType(),True),
        StructField("address",StringType(),True),
        StructField("salary",IntegerType(),True)

    ]
)

data = [
    Row("Raju",32,"Gorakhpur, Uttar Pradesh", 437843),
    Row("Kaju",23,"Pune, Maharashtra", 6564),
    Row("Manju",56,"Latur Maharashtra", 56546),
    Row("Laila",45,"Purnia Bihar", 54354)
        ]

single_row [ Row("Raju",32,"Gorakhpur, Uttar Pradesh", 437843)]

data_rdd = spark.sparkContext.parallelize(data)

data_rdd


NameError: name 'single_row' is not defined

In [ ]:
data_rdd.collect()

[<Row('Raju', 32, 'Gorakhpur, Uttar Pradesh', 437843)>,
 <Row('Kaju', 23, 'Pune, Maharashtra', 6564)>,
 <Row('Manju', 56, 'Latur Maharashtra', 56546)>,
 <Row('Laila', 45, 'Purnia Bihar', 54354)>]

In [ ]:
data_df = spark.createDataFrame(data_rdd, manual_schema)

data_df.show()

+-----+---+--------------------+------+
| name|age|             address|salary|
+-----+---+--------------------+------+
| Raju| 32|Gorakhpur, Uttar ...|437843|
| Kaju| 23|   Pune, Maharashtra|  6564|
|Manju| 56|   Latur Maharashtra| 56546|
|Laila| 45|        Purnia Bihar| 54354|
+-----+---+--------------------+------+



In [ ]:
### Converting to Spark Types (Literals)

from pyspark.sql.functions import lit,expr



df.select(expr("*"), lit(1).alias("one")).show(2)

+-----------------+-------------------+-----+---+
|DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count|one|
+-----------------+-------------------+-----+---+
|    United States|            Romania|    1|  1|
|    United States|            Ireland|  264|  1|
+-----------------+-------------------+-----+---+
only showing top 2 rows


In [ ]:
### Adding Columns


df.withColumn("numberOne",lit(1)).show(2)

+-----------------+-------------------+-----+---------+
|DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count|numberOne|
+-----------------+-------------------+-----+---------+
|    United States|            Romania|    1|        1|
|    United States|            Ireland|  264|        1|
+-----------------+-------------------+-----+---------+
only showing top 2 rows


In [ ]:
## Renamig Columns
df.withColumnRenamed("DEST_COUNTRY_NAME", "dest").columns

['dest', 'ORIGIN_COUNTRY_NAME', 'count']

In [ ]:
from pyspark.sql.functions import col

df.withColumn("count2", col("count").cast("long"))

DataFrame[DEST_COUNTRY_NAME: string, ORIGIN_COUNTRY_NAME: string, count: string, count2: bigint]

In [ ]:
## Filtering column

df.where(col("count") < 2).where(col("ORIGIN_COUNTRY_NAME") != "Croatia")\
.show(2)

+-----------------+-------------------+-----+
|DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count|
+-----------------+-------------------+-----+
|    United States|            Romania|    1|
|Equatorial Guinea|      United States|    1|
+-----------------+-------------------+-----+
only showing top 2 rows


In [ ]:
### getting unique column

df.select("ORIGIN_COUNTRY_NAME", "DEST_COUNTRY_NAME").distinct().count()

320

In [ ]:
df.select("ORIGIN_COUNTRY_NAME").distinct().count()

154

In [ ]:
### Random Samples

seed = 5
withReplacement = False
fraction = 0.5

df.sample(withReplacement, fraction, seed).count()

768

In [ ]:
### sorting

df.sort("count").show(5)
df.orderBy("count", "DEST_COUNTRY_NAME").show(5)
df.orderBy(col("count"), col("DEST_COUNTRY_NAME")).show(5)

+-----------------+-------------------+-----+
|DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count|
+-----------------+-------------------+-----+
|    United States|           Malaysia|    1|
|          Belarus|      United States|    1|
|       The Gambia|      United States|    1|
|            Malta|      United States|    1|
|    United States|            Algeria|    1|
+-----------------+-------------------+-----+
only showing top 5 rows
+-----------------+-------------------+-----+
|DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count|
+-----------------+-------------------+-----+
|       Azerbaijan|      United States|    1|
|          Belarus|      United States|    1|
|          Belarus|      United States|    1|
|           Brunei|      United States|    1|
|         Bulgaria|      United States|    1|
+-----------------+-------------------+-----+
only showing top 5 rows
+-----------------+-------------------+-----+
|DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count|
+-----------------+-------------

In [ ]:
from pyspark.sql.functions import desc, asc
df.orderBy(expr("count desc")).show(2)
df.orderBy(col("count").desc(), col("DEST_COUNTRY_NAME").asc()).show(2)

+-----------------+-------------------+-----+
|DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count|
+-----------------+-------------------+-----+
|    United States|               Togo|    1|
|    United States|    Solomon Islands|    1|
+-----------------+-------------------+-----+
only showing top 2 rows
+-----------------+-------------------+-----+
|DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count|
+-----------------+-------------------+-----+
|           Brazil|      United States|  995|
|    United States|        The Bahamas|  991|
+-----------------+-------------------+-----+
only showing top 2 rows


### sortWithinPartitions

For optimization purposes, it’s sometimes advisable to sort within each partition before another set of transformation

In [ ]:
df = spark.read.format("json").load("../data/flight-data/json/*-summary.json").sortWithinPartitions("count")

In [ ]:
df.limit(10).show()

+--------------------+-------------------+-----+
|   DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count|
+--------------------+-------------------+-----+
|       United States|            Croatia|    1|
|       United States|          Singapore|    1|
|             Moldova|      United States|    1|
|               Malta|      United States|    1|
|       United States|          Gibraltar|    1|
|Saint Vincent and...|      United States|    1|
|            Suriname|      United States|    1|
|       United States|             Cyprus|    1|
|        Burkina Faso|      United States|    1|
|            Djibouti|      United States|    1|
+--------------------+-------------------+-----+



### Repartition and Coalesce

- when the future number of partitions is greater than your current no of partion
- when you are looking to partion by set of columns



In [ ]:
df.rdd.getNumPartitions()

6

In [20]:
df.repartition(5)

DataFrame[DEST_COUNTRY_NAME: string, ORIGIN_COUNTRY_NAME: string, count: string]

In [21]:
df.show()

+--------------------+-------------------+-----+
|   DEST_COUNTRY_NAME|ORIGIN_COUNTRY_NAME|count|
+--------------------+-------------------+-----+
|       United States|            Romania|    1|
|       United States|            Ireland|  264|
|       United States|              India|   69|
|               Egypt|      United States|   24|
|   Equatorial Guinea|      United States|    1|
|       United States|          Singapore|   25|
|       United States|            Grenada|   54|
|          Costa Rica|      United States|  477|
|             Senegal|      United States|   29|
|       United States|   Marshall Islands|   44|
|              Guyana|      United States|   17|
|       United States|       Sint Maarten|   53|
|               Malta|      United States|    1|
|             Bolivia|      United States|   46|
|            Anguilla|      United States|   21|
|Turks and Caicos ...|      United States|  136|
|       United States|        Afghanistan|    2|
|Saint Vincent and..

In [23]:
from pyspark.sql.functions import col

df.repartition(5, col("dest_country_name"))

DataFrame[DEST_COUNTRY_NAME: string, ORIGIN_COUNTRY_NAME: string, count: string]

In [24]:
df.coalesce(2)

DataFrame[DEST_COUNTRY_NAME: string, ORIGIN_COUNTRY_NAME: string, count: string]

In [25]:
collectdf = df.collect()

In [29]:
df.take(5)

[Row(DEST_COUNTRY_NAME='United States', ORIGIN_COUNTRY_NAME='Romania', count='1'),
 Row(DEST_COUNTRY_NAME='United States', ORIGIN_COUNTRY_NAME='Ireland', count='264'),
 Row(DEST_COUNTRY_NAME='United States', ORIGIN_COUNTRY_NAME='India', count='69'),
 Row(DEST_COUNTRY_NAME='Egypt', ORIGIN_COUNTRY_NAME='United States', count='24'),
 Row(DEST_COUNTRY_NAME='Equatorial Guinea', ORIGIN_COUNTRY_NAME='United States', count='1')]

In [30]:
df.limit(5).collect()

[Row(DEST_COUNTRY_NAME='United States', ORIGIN_COUNTRY_NAME='Romania', count='1'),
 Row(DEST_COUNTRY_NAME='United States', ORIGIN_COUNTRY_NAME='Ireland', count='264'),
 Row(DEST_COUNTRY_NAME='United States', ORIGIN_COUNTRY_NAME='India', count='69'),
 Row(DEST_COUNTRY_NAME='Egypt', ORIGIN_COUNTRY_NAME='United States', count='24'),
 Row(DEST_COUNTRY_NAME='Equatorial Guinea', ORIGIN_COUNTRY_NAME='United States', count='1')]

In [ ]:
"""
The method toLocalIterator collects partitions to the driver as an iterator. This
method allows you to iterate over the entire dataset partition-by-partition in a serial manner:


## drawback is this can also crash for large data, it's slow, not parallel
"""

local_iterator = df.limit(10).toLocalIterator() ## same work in scala

local_iterator

<generator object _local_iterator_from_socket.<locals>.PyLocalIterable.__iter__ at 0x70ea87c46400>

In [35]:
for i in local_iterator:
    print(i)

Row(DEST_COUNTRY_NAME='United States', ORIGIN_COUNTRY_NAME='Romania', count='1')
Row(DEST_COUNTRY_NAME='United States', ORIGIN_COUNTRY_NAME='Ireland', count='264')
Row(DEST_COUNTRY_NAME='United States', ORIGIN_COUNTRY_NAME='India', count='69')
Row(DEST_COUNTRY_NAME='Egypt', ORIGIN_COUNTRY_NAME='United States', count='24')
Row(DEST_COUNTRY_NAME='Equatorial Guinea', ORIGIN_COUNTRY_NAME='United States', count='1')
Row(DEST_COUNTRY_NAME='United States', ORIGIN_COUNTRY_NAME='Singapore', count='25')
Row(DEST_COUNTRY_NAME='United States', ORIGIN_COUNTRY_NAME='Grenada', count='54')
Row(DEST_COUNTRY_NAME='Costa Rica', ORIGIN_COUNTRY_NAME='United States', count='477')
Row(DEST_COUNTRY_NAME='Senegal', ORIGIN_COUNTRY_NAME='United States', count='29')
Row(DEST_COUNTRY_NAME='United States', ORIGIN_COUNTRY_NAME='Marshall Islands', count='44')


In [36]:
spark.stop()